# Vocabulary Richness Beyond TTR

**Computational Text Analysis Pack · Module 02 — Describing a Corpus**

*Technique:* Yule's K, Herdan's C, MATTR and Heaps's law; length-robust richness

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/02-describing-corpora/02e_vocabulary_richness.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import math

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Notebook 02c showed that type/token ratio falls as a text gets longer, which
makes it useless for comparing texts of different lengths — and almost every
real corpus has texts of different lengths.

That leaves an obvious question the course never answered: **what should you use
instead?** This notebook computes the four standard answers and shows which of
them actually survive the length problem.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
# One row per work, so we have documents of genuinely different lengths.
works = (df_colonial.groupby(["work_id", "title", "author"])["text"]
                    .apply(" ".join).reset_index())
works["tokens"] = works["text"].map(tokenise)
works["N"] = works["tokens"].map(len)
works = works[works["N"] >= 2000].reset_index(drop=True)
print(f"{len(works)} works, {works['N'].min():,} to {works['N'].max():,} tokens")

## The four measures

- **TTR** — types divided by tokens. The baseline, and length-dependent.
- **Herdan's C** — `log V / log N`. Takes logs of both, which flattens but does
  not remove the length effect.
- **Yule's K** — built from how many words occur once, twice, three times. It is
  a property of the *shape* of the frequency distribution, which is why it is
  far more stable than TTR.
- **MATTR** — moving-average TTR: slide a fixed window across the text and
  average. Because every window is the same size, length cancels out by
  construction.

In [ ]:
def ttr(toks):
    return len(set(toks)) / len(toks)

def herdan_c(toks):
    return math.log(len(set(toks))) / math.log(len(toks))

def yule_k(toks):
    # V_i = how many word types occur exactly i times.
    spectrum = Counter(Counter(toks).values())
    N = len(toks)
    inner = sum(i * i * vi for i, vi in spectrum.items()) - N
    return 10_000 * inner / (N * N)

def mattr(toks, window=500):
    if len(toks) <= window:
        return ttr(toks)
    scores = [len(set(toks[i:i + window])) / window
              for i in range(0, len(toks) - window, window // 2)]
    return sum(scores) / len(scores)

works["TTR"] = works["tokens"].map(ttr)
works["Herdan_C"] = works["tokens"].map(herdan_c)
works["Yule_K"] = works["tokens"].map(yule_k)
works["MATTR"] = works["tokens"].map(mattr)

print(works[["title", "N", "TTR", "Herdan_C", "Yule_K", "MATTR"]]
      .head(8).to_string(index=False,
                         float_format=lambda v: f"{v:.3f}"))

## Which of them is actually independent of length?

The test is simple: correlate each measure against text length. A measure that
still tracks length has not solved the problem, it has only disguised it.

In [ ]:
print(f"{'measure':<12}{'correlation with length':>26}")
print("-" * 38)
for col in ["TTR", "Herdan_C", "Yule_K", "MATTR"]:
    r = works[col].corr(works["N"])
    verdict = "length-dependent" if abs(r) > 0.3 else "usable"
    print(f"{col:<12}{r:>18.3f}   {verdict}")
print()
print("TTR and Herdan's C still slide with length. Yule's K and MATTR do not,")
print("which is why they are the ones to report when texts differ in size.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(works["N"], works["TTR"], s=18)
axes[0].set_xlabel("tokens")
axes[0].set_ylabel("TTR")
axes[0].set_title("TTR: slides with length")
axes[1].scatter(works["N"], works["MATTR"], s=18, color="darkorange")
axes[1].set_xlabel("tokens")
axes[1].set_ylabel("MATTR")
axes[1].set_title("MATTR: flat")
plt.tight_layout()

## Heaps's law

Zipf's law (notebook 02b) describes how often words repeat. **Heaps's law**
describes the other side of the same coin: how fast *new* words arrive as a text
grows, as `V = K · N^β`. Fitting β on a log-log plot gives a number you can
compare across languages and genres — and it explains why TTR had to fall.

In [ ]:
big = works.loc[works["N"].idxmax(), "tokens"]
steps = range(1000, len(big), 1000)
seen, V, curve = set(), 0, []
for end in steps:
    for w in big[end - 1000:end]:
        seen.add(w)
    curve.append((end, len(seen)))

xs = np.array([c[0] for c in curve], dtype=float)
ys = np.array([c[1] for c in curve], dtype=float)
beta, log_k = np.polyfit(np.log(xs), np.log(ys), 1)
print(f"Heaps's law fit:  V = {math.exp(log_k):.2f} * N^{beta:.3f}")
print()
print("beta below 1 means vocabulary grows more slowly than text length.")
print("That IS the TTR problem, stated as a law rather than a nuisance.")

plt.figure(figsize=(6, 4))
plt.loglog(xs, ys, marker="o", ms=3, lw=1, label="observed")
plt.loglog(xs, math.exp(log_k) * xs ** beta, "--", label=f"fit, beta={beta:.3f}")
plt.xlabel("tokens (N)")
plt.ylabel("types (V)")
plt.legend()

### 🔧 Your turn 1

Change the MATTR window from 500 to 100 and re-run. Does the correlation with length stay near zero? What does a very small window measure that a large one does not?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

A 100-word window still cancels length, so the correlation stays low — but the
values rise sharply and compress together, because almost every 100-word window
of running prose is mostly distinct words. A small window measures *local*
variety and loses the ability to distinguish authors; a large one measures the
text's overall lexical range. Covington & McFall's original recommendation is a
window in the low hundreds, and reporting the window is not optional: MATTR
without its window size is an uninterpretable number.

</details>